In [0]:

# Task 3: Data Center IoT Gold Summary
# Silver Delta -> Gold Delta -> SQL Table -> CSV Report
# Shared process_date from Databricks Job

from datetime import datetime

from pyspark.sql.functions import (
    col,
    count,
    avg,
    when,
    max as spark_max,
    sum as spark_sum,
    round as spark_round
)

from pipeline_logger import logger, save_logs_to_volume


try:
    logger.info("===== DATA CENTER GOLD STARTED =====")

    base = "/Volumes/workspace/iot_db/data"

    # STEP 1: Get shared process date from Job
    # Job parameter:
    # {{tasks.00_prepare_date.values.process_date}}

    dbutils.widgets.text(
        "process_date",
        "",
        "Process Date"
    )

    process_date = dbutils.widgets.get(
        "process_date"
    ).strip()

    if not process_date:
        raise ValueError(
            "process_date is required from 00_prepare_date task"
        )

    # Validate date format
    datetime.strptime(process_date, "%Y-%m-%d")

    print("Process Date:", process_date)
    logger.info(f"Processing date: {process_date}")

    # STEP 2: Read clean Silver data
    silver_path = f"{base}/dc_silver"

    df = (
        spark.read
        .format("delta")
        .load(silver_path)
        .filter(col("process_date") == process_date)
    )

    silver_count = df.count()

    print("Silver records:", silver_count)

    if silver_count == 0:
        raise ValueError(
            f"No Silver records found for {process_date}"
        )

    # STEP 3: Create Data Center summary
    gold_df = df.groupBy(
        "data_center_id",
        "city",
        "region",
        "process_date"
    ).agg(

        count("*").alias("total_readings"),

        spark_round(
            avg("temperature_c"), 2
        ).alias("average_temperature"),

        spark_max(
            "temperature_c"
        ).alias("maximum_temperature"),

        spark_round(
            avg("humidity_percent"), 2
        ).alias("average_humidity"),

        spark_round(
            avg("power_consumption_kw"), 2
        ).alias("average_power_kw"),

        spark_sum(
            when(col("alert_level") == "CRITICAL", 1)
            .otherwise(0)
        ).alias("critical_readings"),

        spark_sum(
            when(col("alert_level") == "WARNING", 1)
            .otherwise(0)
        ).alias("warning_readings"),

        spark_sum(
            when(col("cooling_status") == "FAILED", 1)
            .otherwise(0)
        ).alias("cooling_failures"),

        spark_sum(
            when(col("cooling_status") == "DEGRADED", 1)
            .otherwise(0)
        ).alias("cooling_degraded_readings"),

        spark_sum(
            when(col("sensor_status") == "OFFLINE", 1)
            .otherwise(0)
        ).alias("offline_sensor_readings")

    ).orderBy("data_center_id")

    # STEP 4: Display Gold summary
    display(gold_df)

    gold_count = gold_df.count()

    print("Data Centers:", gold_count)

    # STEP 5: Save Gold Delta in Volume
    gold_path = f"{base}/dc_gold"

    (
        gold_df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"process_date = '{process_date}'"
        )
        .partitionBy("process_date")
        .save(gold_path)
    )

    logger.info(f"Gold Delta saved: {gold_path}")

    # STEP 6: Save Gold SQL table for dashboard
    gold_table = "workspace.iot_db.gold_dc_summary"

    if spark.catalog.tableExists(gold_table):

        # Delete only selected date's previous summary
        spark.sql(
            f"DELETE FROM {gold_table} "
            f"WHERE process_date = '{process_date}'"
        )

        (
            gold_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(gold_table)
        )

    else:

        (
            gold_df.write
            .format("delta")
            .partitionBy("process_date")
            .saveAsTable(gold_table)
        )

    logger.info(f"Gold table saved: {gold_table}")

    # STEP 7: Create output folder
    output_dir = f"{base}/output"

    dbutils.fs.mkdirs(output_dir)

    # STEP 8: Save daily CSV report
    output_file = (
        f"{output_dir}/"
        f"dc_gold_summary_{process_date}.csv"
    )

    gold_df.toPandas().to_csv(
        output_file,
        index=False
    )

    logger.info(f"CSV report saved: {output_file}")

    # STEP 9: Success message
    print("DATA CENTER GOLD COMPLETED SUCCESSFULLY")
    print("Process Date:", process_date)
    print("Data Centers:", gold_count)
    print("Gold Table:", gold_table)
    print("Gold Delta Path:", gold_path)
    print("CSV Report:", output_file)

    logger.info(
        "Data Center Gold completed successfully"
    )

except Exception:
    logger.exception("Data Center Gold generation failed")
    raise

finally:
    save_logs_to_volume(dbutils)


Silver records: 1409727


data_center_id,city,region,process_date,total_readings,average_temperature,maximum_temperature,average_humidity,average_power_kw,critical_readings,warning_readings,cooling_failures,cooling_degraded_readings,offline_sensor_readings
DC01,Mumbai,WEST,2026-10-18,140890,23.94,42.0,50.02,8.0,1359,8314,1359,5587,1453
DC02,Navi Mumbai,WEST,2026-10-18,141175,23.92,42.0,50.02,8.0,1368,8292,1368,5638,1367
DC03,Chennai,SOUTH,2026-10-18,140989,23.92,42.0,50.04,8.0,1363,8306,1363,5582,1463
DC04,Hyderabad,SOUTH,2026-10-18,140902,23.93,42.0,49.99,8.0,1408,8345,1408,5593,1407
DC05,Bengaluru,SOUTH,2026-10-18,140954,23.94,42.0,49.97,7.99,1370,8226,1370,5560,1432
DC06,Noida,NORTH,2026-10-18,140964,23.94,42.0,50.03,7.99,1449,8139,1449,5495,1359
DC07,Greater Noida,NORTH,2026-10-18,141015,23.94,42.0,50.02,7.99,1390,8299,1390,5633,1400
DC08,Pune,WEST,2026-10-18,140951,23.94,42.0,50.01,8.0,1417,8315,1417,5627,1405
DC09,Kolkata,EAST,2026-10-18,140897,23.95,42.0,50.0,8.0,1364,8417,1364,5705,1415
DC10,Ahmedabad,WEST,2026-10-18,140990,23.92,42.0,49.98,8.0,1380,8449,1380,5712,1444


Data Centers: 10
DATA CENTER GOLD COMPLETED SUCCESSFULLY
Gold table: workspace.iot_db.gold_dc_summary
Gold path: /Volumes/workspace/iot_db/data/dc_gold
CSV report: /Volumes/workspace/iot_db/data/output/dc_gold_summary_2026-10-18.csv
Logs saved to /Volumes/workspace/iot_db/data/logs/2026-10-18_app.log
